# Chapter5 : 오차역전파법 (backpropagation)
# 5.1 계산 그래프 (computational graph)

- 계산과정을 표현한 그래프
- 복수의 노드(node)와 엣지(edge)로 표현됨

## 5.1.1 계산그래프로 풀다
문제 : 현빈 군은 슈퍼에서 1개에 100원인 사과를 2개 샀습니다. 이때 지불 금액을 구하세요. 단, 소비세가 10% 부과됩니다.

> 100 ----> x2 ----> x1.1 ----> 220

사과의 100원이 x2노드로 흐름, 200원이 되어 x1.1노드로 흐름 최종적으로 220원

- 순전파 (forward propagation) : 출발점에서 도착점으로
- 역전파 (propagation) : 순전파 반대 방향

## 5.1.2 국소적 계산

- 전체는 복잡하지만 계산 그래프에서 한 노드는 국소적 계산(자신과만 관련있는 계산)을 함

## 5.1.3 왜 계산 그래프로 푸는가?

- 순전파와 역전파를 이용해서 각 변수의 미분을 효율적으로 구할 수 있음



# 5.2 연쇄법칙 (chain rule)
- 국소적 미분을 전달하는 원리

## 5.2.1 계산 그래프의 역전파
> --$x$--> $f$ --$y$-->
>
> <--$E \frac{\partial y}{\partial x}$-- $f$ <--$E$--

- 신호 E에 노드의 국소적 미분 ($\frac{\partial y}{\partial x}$)을 곱한후 다음 노드로 전달하여 역전파를 계산

- x에대한 y의 미분

## 5.2.2 연쇄법칙이란?

### 합성함수 - 여러 함수로 구성된 함수

- 예
$$z = (x + y)$$
$$z = t^2$$
$$t = x + y$$

- 합성함수의 미분은 합성함수를 구성하는 각 함수의 미분의 곱으로 나타낼 수 있다.

$$\frac{\partial z}{\partial x} = \frac{\partial z}{\partial t}\frac{\partial t}{\partial x}$$

$$\frac{\partial z}{\partial t} = 2t$$
$$\frac{\partial t}{\partial x} = 1$$

$$\frac{\partial z}{\partial x} = \frac{\partial z}{\partial t}\frac{\partial t}{\partial x} = 2t * 1 = 2(x+y)$$

## 5.2.3 연쇄법칙과 계산 그래프
> --$x$--> $(+)$ --$t$--> $(**2)$ --$z$-->
>
><--$\frac{\partial z}{\partial z}\frac{\partial z}{\partial t}\frac{\partial t}{\partial x}$-- $(+)$ <--$\frac{\partial z}{\partial z}\frac{\partial z}{\partial t}$-- $(**2)$ <--$\frac{\partial z}{\partial z}$--

역전파가 z에서 t로 간다면 미분은 z에 대한 t의 미분


# 5.3 역전파
## 5.3.1 덧셈 노드의 역전파
$$z = x+y$$
$$\frac{\partial z}{\partial x} = 1$$
$$\frac{\partial z}{\partial y} = 1$$

덧셈노드의 역전파는 걍 1임

## 5.3.2 곱셈 노드의 역전파
$$z = xy$$
$$\frac{\partial z}{\partial x} = y$$
$$\frac{\partial z}{\partial y} = x$$

- 예
```
--10--> x --50-->
--5-----|

<--6.5-- x <--1.3--
<--13----|
```

# 5.4 단순한 계층 구현하기
## 5.4.1 곱셈 계층
### MulLayer 클래스

- _ _init _ _() : 인스턴스 변수인 x와 y를 초기화 함.
- forward() : 순전파 - x,y를 받아서 곱해서 반환
- backward() : 역전파 - 상류에서 넘어온 미분(dout)에 순전파 때의 값을 서로 바꿔 곱한뒤 하류로 흘림

In [79]:
class MulLayer:
  def __init__(self):
    self.x = None
    self.y = None

  def forward(self, x, y):
    self.x = x
    self.y = y
    out = x * y

    return out

  def backward(self, dout):
    dx = dout * self.y # x와 y를 바꾼다. 이게 왜 되지?
    dy = dout * self.x

    return dx, dy

- 사과 2개 구입 예제
- 가격 100원
- 갯수 2개
- 세금 1.1

In [80]:
apple = 100
apple_num = 2
tax = 1.1

# 계층들
mul_apple_layer = MulLayer()
mul_tax_layer = MulLayer()

# 순전파
apple_price = mul_apple_layer. forward(apple, apple_num)
price = mul_tax_layer. forward(apple_price, tax)

print(price) # 220

220.00000000000003


In [81]:
# 역전파
dprice= 1
dapple_price, dtax = mul_tax_layer.backward(dprice)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)

print(dapple, dapple_num, dtax) # 2.2 110 200

2.2 110.00000000000001 200


## 5.4.2 덧셈계층
### ADDLayer 클래스
- _ _ init _ _() : 초기화 필요없음
- forward() : 그냥 x,y를 더해서 반환
- backward() : 1곱해서 반환

In [82]:
class AddLayer:
  def __init__(self):
    pass

  def forward(self, x, y):
    out = x + y
    return out

  def backward(self, dout):
    dx = dout * 1
    dy = dout * 1
    return dx, dy

사과 2개와 귤 3개 구입

In [83]:
apple = 100
apple_num = 2
orange = 150
orange_num = 3
tax = 1.1

# 계층들
mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_apple_orange_layer = AddLayer()
mul_tax_layer = MulLayer()

# 순전파
apple_price = mul_apple_layer. forward(apple, apple_num)
orange_price = mul_orange_layer. forward(orange, orange_num)
all_price = add_apple_orange_layer. forward(apple_price, orange_price)
price = mul_tax_layer. forward(all_price, tax)

# 역전파
dprice = 1
dall_price, dtax = mul_tax_layer.backward(dprice)
dapple_price, dorange_price = add_apple_orange_layer.backward(dall_price)
dorange, dorange_num = mul_orange_layer.backward(dorange_price)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)

print(price) # 715
print(dapple_num, dapple, dorange, dorange_num, dtax) # 110 2.2 3.3 165 650

715.0000000000001
110.00000000000001 2.2 3.3000000000000003 165.0 650


# 5.5 활성화 함수 계층 구현하기
## 5.5.1 ReLU 계층
### ReLU의 수식
$$ y = \begin{cases}
x & (x > 0)\\
0 & (x \le 0)
\end{cases}$$

$$ \frac{\partial y}{\partial x}
= \begin{cases}
1 & (x > 0)\\
0 & (x \le 0)
\end{cases}$$

- 순전파때의 입력인 x가 0보다 크면 역전파는 상류의 값을 그대로 하류로 보냄
- x가 0보다 작거나 같으면 역전파는 하류로 신호를 보내지 않음

### ReLu 클래스
- mask 인스턴스 변수 : True/False로 구성된 넘파이 배열
  - 순전파의 입력인 x의 원소값이 0이하인 인덱스는 True
  - 그외는 False
- backward에서 mask의 원소가 True이면 dout이 0

In [84]:
class Relu:
  def __init__(self):
    self.mask = None

  def forward(self, x):
    self.mask = (x <= 0)
    out = x.copy()
    out[self.mask] = 0

    return out

  def backward(self, dout):
    dout[self.mask] = 0
    dx = dout

    return dx

In [85]:
import numpy as np

x = np.array([[1.0,-0.5], [-2.0, 3.0]] )
print(x)

mask = (x <= 0)
print(mask)

[[ 1.  -0.5]
 [-2.   3. ]]
[[False  True]
 [ True False]]


## 5.5.2 Sigmoid 계층
### 시그모이드 함수
$$y = \frac{1}{1+exp(-x)}$$

### 시그모이드 함수 역전파의 흐름
- 1단계 : '/'노드, 즉 $y = \frac{1}{x}$ 미분
$$\frac{\partial y}{\partial x} = -\frac{1}{x^2} = -y^2$$
역전파때 상류에서 흘러온 값에 $-y^2$곱해서 하류로 전달

- 2단계 : '+'노드, 상류의 값을 여과없이 걍 하류로 전달

- 3단계 : 'exp'노드, $y = exp(x)$ 연산 수행
$$\frac{\partial y}{\partial x} = exp(x)$$
계산그래프에서 $exp(-x)$ 곱해서 전달

-4단계 : 'x'노드는 순전파때의 값을 서로바꿔 곱함. 이 예시에서는 -1을 곱함

- 간소화
$$\frac{\partial L}{\partial y} ->
\frac{\partial L}{\partial y}y^2 exp(-x)
$$


In [86]:
class Sigmoid:
  def __init__(self):
    self.out = None

  def forward(self, x):
    out =1/(1+np.exp(-x))
    self.out = out

    return out

  def backward(self, dout):
    dx = dout * (1.0- self.out) * self.out

    return dx

# 5.6 Affine / Softmax 게층 구현하기
## 5.6.1 Affine 계층

신경망의 순전파 때 수행하는 행렬의 곱은 기하학에서 어파인 변환 (affine transformation) 이라고함

- Y = np.dot(X,W) + B

### 행렬을 사용한 역전파

$$ \frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} \cdot W^T $$

$$ \frac{\partial L}{\partial W} = X^T \cdot \frac{\partial L}{\partial Y}$$

### Affine 클래스
- 편향의 역전파는 그 두 데이터에 대한 미분을 데이터마다 더해서 구함

In [87]:
class Affine:
  def __init__(self, W, b):
    self.W = W
    self.b = b
    self.x = None
    self.dW = None
    self.db = None

  def forward(self, x):
    self.x = x
    out = np.dot(x, self.W) + self.b

    return out

  def backward(self, dout):
    dx = np.dot(dout, self.W.T)
    self.dW = np.dot(self.x.T, dout)
    self.db = np.sum(dout, axis=0)

    return dx

## 5.6.3 Softmax-with-Loss 계층
소프트 맥스 함수는 출력층에서 사용
- 출력을 정규화하는 특징이 있음

- 소프트 맥스 계층을 교차 엔트로피 오차도 함께 포함하여 만듬 -> Softmax-with-Loss

### Softmax 계층의 역전파
$$(y_1 - t_1, y_2 - t_2, y_3 - t_3)$$

- Softmax 계층의 출력 : $(y_1, y_2, y_3)$
- 정답 레이블 : $(t_1, t_2, t_3)$

In [88]:
def softmax(a):
  c = np.max(a)
  exp_a = np.exp(a - c) # 오버플로우 방지
  sum_exp_a = np.sum(exp_a)
  y = exp_a / sum_exp_a
  return y

In [89]:
def cross_entropy_error(y, t):
  if y.ndim == 1:
    t = t.reshape(1, t.size)
    y = y.reshape(1, y.size)

  batch_size = y. shape[0]
  return -np.sum(t * np.log(y + 1e-7)) / batch_size

In [90]:
class SoftmaxWithLoss:
  def __init__(self):
    self.loss = None  # 손실함수
    self.y = None     # softmax의 출력
    self.t = None     # 정답 레이블(원-핫 벡터)

  def forward(self, x, t):
    self.t = t
    self.y = softmax(x)
    self.loss = cross_entropy_error(self.y, self.t)

    return self.loss

  def backward(self, dout=1):
    batch_size = self. t.shape[0]
    dx = (self.y - self.t) / batch_size

    return dx

# 5.7 오차역전파법 구현하기
## 5.7.1 신경망 학습의 전체 그림
### 전체
- 신경망에는 적용가능한 가중치와 편향이 있고, 이를 훈련 데이터에 적응하도록 조정하는 과정을 학습이라고 함.
### 1단계 - 미니배치
- 훈련 데이터 중 일부를 무작위로 가져옴
### 2단계 - 기울기 산출
- 미니배치의 손실 한수 값을 줄이기 위해 각 가중치 매개변수의 기울기를 구함. 기울기는 손실함수의 값을 가장 작게 하는 방향을 제시

### 3단계 - 매개변수 갱신
- 가중치 매개변수를 기울기 방향으로 아주 조금 갱신

### 4단계 - 반복
- 1~3단계를 반봅

## 5.7.2 오차역전파법을 적용한 신경망 구현하기
- 여기에서는 2층 신경망을 TwoLayerNet 클래스로 구현

In [91]:
!git clone https://github.com/WegraLee/deep-learning-from-scratch.git
%cd deep-learning-from-scratch

Cloning into 'deep-learning-from-scratch'...
remote: Enumerating objects: 1118, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 1118 (delta 141), reused 130 (delta 130), pack-reused 943 (from 2)
Receiving objects: 100% (1118/1118), 52.68 MiB | 38.18 MiB/s, done.
Resolving deltas: 100% (644/644), done.
/content/deep-learning-from-scratch/deep-learning-from-scratch/deep-learning-from-scratch/deep-learning-from-scratch


In [92]:
import numpy as np

from common.layers import *
from common.gradient import numerical_gradient
from collections import OrderedDict

class TwoLayerNet:
  def __init__(self, input_size, hidden_size, output_size,weight_init_std=0.01):
    # 가중치 초기화
    self.params = {}
    self.params['W1'] = weight_init_std * \
                        np.random.randn(input_size, hidden_size)
    self.params['b1'] = np.zeros(hidden_size)
    self.params['W2'] = weight_init_std * \
                        np.random.randn(hidden_size, output_size)
    self.params['b2'] = np.zeros(output_size)

    # 계층 생성
    self.layers = OrderedDict() # 1
    self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
    self.layers['Relu1'] = Relu()
    self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])

    self.lastLayer = SoftmaxWithLoss()

  def predict(self, x):
    for layer in self.layers.values(): # 2
      x = layer.forward(x)

    return x

  # x: 입력 데이터, t: 정답 레이블
  def loss(self, x, t):
    y = self.predict(x)
    return self.lastLayer. forward(y, t)

  def accuracy(self, x, t):
    y = self.predict(x)
    y = np.argmax(y, axis=1)
    if t.ndim != 1 : t = np.argmax(t, axis=1)

    accuracy = np.sum(y == t) / float(x. shape[0])

    return accuracy

  # x: 입력 데이터, t: 정답 레이블
  def numerical_gradient(self, x, t):
    loss_W = lambda W: self.loss(x, t)

    grads = {}
    grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
    grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
    grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
    grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

    return grads

  def gradient(self, x, t):
    # 순전파
    self.loss(x, t)

    # 역전파
    dout = 1
    dout = self.lastLayer.backward(dout)

    layers = list(self.layers.values())
    layers.reverse() # 3
    for layer in layers:
      dout = layer.backward(dout)

    # 결과 저장
    grads = {}
    grads['W1'] = self.layers['Affine1'].dW
    grads['b1'] = self.layers['Affine1'].db
    grads['W2'] = self.layers['Affine2'].dW
    grads['b2'] = self.layers['Affine2'].db

    return grads

- 신경망의 계층을 OrderedDict에 보관
- 순전파 때는 순서대로 forward()메서드를 호출하면됨
- 역전파는 거꾸로

## 5.7.3 오차역전파법으로 구한 기울기 검증하기

### 기울기 확인 (gradient check)
- 수치 미분과 오차역전파법의 결과를 비교
- 오차역전파법을 제대로 구현했는지를 검증

In [93]:
import numpy as np
from dataset.mnist import load_mnist

# 데이터 읽기
(x_train, t_train), (x_test, t_test) = \
    load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

x_batch = x_train[:3]
t_batch = t_train[ :3]

grad_numerical = network.numerical_gradient(x_batch, t_batch)
grad_backprop = network.gradient(x_batch, t_batch)

# 각 가중치의 차이의 절댓값을 구한 후, 그 절댓값들의 평균을 낸다.
for key in grad_numerical.keys():
  diff = np.average( np.abs(grad_backprop[key] - grad_numerical[key]) )
  print(key + ":" + str(diff))

W1:3.926265354416184e-10
b1:2.0025300594559807e-09
W2:5.29228068689923e-09
b2:1.398338087807427e-07


## 5.7.4 오차역전파법을 사용한 학습 구현하기


In [95]:
import numpy as np
from dataset.mnist import load_mnist

# 데이터 읽기
(x_train, t_train), (x_test, t_test) = \
  load_mnist(normalize=True, one_hot_label=True)
network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iters_num):
  batch_mask = np.random.choice(train_size, batch_size)
  x_batch = x_train[batch_mask]
  t_batch = t_train[batch_mask]

  # 오차역전파법으로 기울기를 구한다.
  grad = network.gradient(x_batch, t_batch)

  # 갱신
  for key in ('W1', 'b1', 'W2', 'b2'):
    network.params [key] -= learning_rate * grad[key]

  loss = network. loss(x_batch, t_batch)

  train_loss_list. append(loss)

  if i % iter_per_epoch == 0:
    train_acc = network. accuracy(x_train, t_train)
    test_acc = network. accuracy(x_test, t_test)
    train_acc_list.append(train_acc)
    test_acc_list. append(test_acc)
    print(train_acc, test_acc)

0.1289 0.1394
0.9030666666666667 0.9074
0.92385 0.9263
0.9307 0.9293
0.9432666666666667 0.9407
0.9510166666666666 0.9481
0.95465 0.9495
0.9599833333333333 0.9557
0.9643166666666667 0.96
0.9659166666666666 0.9619
0.9686 0.9633
0.9709333333333333 0.9662
0.9731166666666666 0.9655
0.9746166666666667 0.9663
0.9760333333333333 0.9675
0.9770166666666666 0.9682
0.97775 0.968


# 5.8 정리
- 계산 그래프를 이용해 신경망의 동작과 오차역전파법 설명
- 그 처리 과정을 계층으로 설명
- ReLU 계층, Softmax-with-Loss 계층, Affine 계층, Softmax 계층
- 모든 계층에서 forward와 backward라는 메서드 구현